# Notebook 4: Confidence Intervals and Bootstrap Methods

## Overview
In this notebook, we explore **confidence intervals** and **bootstrap resampling**—two powerful tools for quantifying uncertainty in A/B testing results.

### Learning Objectives
- Understand what confidence intervals really mean (and common misconceptions)
- Calculate confidence intervals analytically for proportions and means
- Learn the bootstrap principle: resampling to estimate uncertainty
- Implement bootstrap hypothesis tests
- Compare analytical vs. bootstrap approaches

### Why This Matters
A/B test results should always include uncertainty estimates. A point estimate alone (e.g., "conversion rate is 5%") is incomplete. Confidence intervals tell us the range of plausible values and help us make rigorous decisions.

**Key Insight**: The bootstrap is a practical, flexible method that works for almost any statistic, even when we don't have a formula for its distribution.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try to import plotly for optional interactive plots
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

# Set style
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb04', exist_ok=True)


In [ ]:
# Load cleaned data
data_path = Path("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"\nFirst rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nGroup distribution:\n{df['segment'].value_counts()}")

## Concept: What Are Confidence Intervals?

### The (Misunderstood) Idea
A **confidence interval** is a range of values computed from sample data, designed to bracket an unknown population parameter.

#### Common Misconception
Many people think a 95% CI means "there's a 95% probability the true parameter is in this interval." **This is wrong!**

The true parameter is either in the interval or it isn't—there's no probability. Instead, 95% CI means:

> If we repeated our experiment many times, ~95% of the intervals we construct would contain the true parameter.

It's a **long-run frequency** property, not a direct probability about this particular interval.

#### Types of Confidence Intervals
1. **Wald Interval** (Normal approximation): Simple, but can be inaccurate for proportions near 0 or 1
2. **Wilson Score Interval**: More accurate, especially for small samples and extreme proportions
3. **Bootstrap Percentile**: Uses resampling, works for any statistic
4. **Highest Density Interval (HDI)**: Bayesian approach (covered in Notebook 6)

#### In A/B Testing
We usually care about:
- Confidence intervals for **conversion rate difference** (e.g., Men's - Control)
- Confidence intervals for **average spend difference**
- Whether the CI excludes zero (suggesting a real difference)

## Analytical Confidence Intervals

### Wald and Wilson Intervals for Proportions
For a single proportion p, we can calculate CIs analytically.
- **Wald CI**: Uses the normal approximation; simple but unreliable for extreme p
- **Wilson CI**: Uses score inversion; more robust

For the **difference** between two proportions, we typically use the Wald method with a continuity correction.

In [ ]:
def wald_ci_proportion(successes, n, confidence=0.95):
    """
    Calculate Wald confidence interval for a proportion.
    
    Parameters:
    -----------
    successes : int
        Number of successes
    n : int
        Total sample size
    confidence : float
        Confidence level (default 0.95)
    
    Returns:
    --------
    tuple : (point_estimate, lower_bound, upper_bound)
    """
    p = successes / n
    se = np.sqrt(p * (1 - p) / n)
    z = stats.norm.ppf((1 + confidence) / 2)
    
    lower = p - z * se
    upper = p + z * se
    
    return p, max(0, lower), min(1, upper)

def wilson_ci_proportion(successes, n, confidence=0.95):
    """
    Calculate Wilson score confidence interval for a proportion.
    More robust than Wald, especially for extreme proportions.
    """
    p = successes / n
    z = stats.norm.ppf((1 + confidence) / 2)
    z_sq = z ** 2
    
    denom = 1 + z_sq / n
    center = (p + z_sq / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z_sq / (4 * n ** 2)) / denom
    
    return p, center - margin, center + margin

def ci_difference_proportions(succ1, n1, succ2, n2, confidence=0.95):
    """
    Wald CI for difference between two proportions: p1 - p2
    """
    p1 = succ1 / n1
    p2 = succ2 / n2
    diff = p1 - p2
    
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    z = stats.norm.ppf((1 + confidence) / 2)
    
    return diff, diff - z * se, diff + z * se

# Calculate conversion rate for each segment
segment_stats = df.groupby('segment').agg({
    'conversion': ['sum', 'count', 'mean']
}).round(4)

segment_stats.columns = ['Conversions', 'N', 'Rate']
print("Conversion Statistics by Segment:")
print(segment_stats)
print()

# Calculate CIs for conversion rates
groups = df['segment'].unique()
ci_results = []

for group in groups:
    group_data = df[df['segment'] == group]
    conv_count = group_data['conversion'].sum()
    total = len(group_data)
    
    # Wald CI
    p_wald, lower_wald, upper_wald = wald_ci_proportion(conv_count, total)
    
    # Wilson CI
    p_wilson, lower_wilson, upper_wilson = wilson_ci_proportion(conv_count, total)
    
    ci_results.append({
        'Segment': group,
        'N': total,
        'Conversions': conv_count,
        'Rate': p_wald,
        'Wald Lower': lower_wald,
        'Wald Upper': upper_wald,
        'Wilson Lower': lower_wilson,
        'Wilson Upper': upper_wilson
    })

ci_df = pd.DataFrame(ci_results)
print("Confidence Intervals for Conversion Rates (95%):")
print(ci_df.to_string(index=False))

In [ ]:
# Visualize CIs for conversion rates (forest plot style)
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(ci_df))
groups_display = ci_df['Segment'].values
means = ci_df['Rate'].values
lower_wald = ci_df['Wald Lower'].values
upper_wald = ci_df['Wald Upper'].values
lower_wilson = ci_df['Wilson Lower'].values
upper_wilson = ci_df['Wilson Upper'].values

# Plot Wald intervals
ax.barh(y_pos - 0.15, upper_wald - lower_wald, left=lower_wald, height=0.3, 
        label='Wald CI', alpha=0.7, color='steelblue')

# Plot Wilson intervals
ax.barh(y_pos + 0.15, upper_wilson - lower_wilson, left=lower_wilson, height=0.3, 
        label='Wilson CI', alpha=0.7, color='darkorange')

# Plot point estimates
ax.scatter(means, y_pos, color='black', s=100, zorder=5, label='Point Estimate')

ax.set_yticks(y_pos)
ax.set_yticklabels(groups_display)
ax.set_xlabel('Conversion Rate')
ax.set_title('95% Confidence Intervals for Conversion Rates\nWald vs Wilson Methods', 
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb04/nb04_ci_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("Forest plot saved.")

## Concept: The Bootstrap

### The Big Idea
The **bootstrap** is a resampling method that uses your sample to estimate the distribution of a statistic without making strong assumptions.

### How It Works (Intuitive Example)
Imagine you have a bag of 100 marbles from a factory. You draw a random sample of 10 marbles. The bootstrap asks:

> "If I repeatedly draw samples (with replacement) from these 10 marbles, what can I learn about the distribution of the sample proportion?"

The intuition: **the sample is a mini-population**. By resampling from it, we can estimate the sampling distribution of our statistic.

### The Algorithm (Percentile Bootstrap)
1. **Original sample**: Your data
2. **Resample**: Draw n observations **with replacement** from your data (n times)
3. **Calculate statistic**: Compute your statistic (mean, median, difference, etc.) on the resample
4. **Repeat steps 2-3** many times (typically 10,000+)
5. **Percentiles**: Use the 2.5th and 97.5th percentiles of the bootstrap distribution as 95% CI

### Why It Works
The bootstrap relies on a key insight: if your sample is representative of the population, then the sample is representative of the population in the same way the bootstrap resamples are representative of the sample.

### Advantages
- **Works for any statistic**: mean, median, ratio, difference, correlation, etc.
- **No distributional assumptions**: doesn't assume normality
- **Handles small samples**: sometimes more accurate than theory-based methods
- **Flexible**: can implement complex sampling designs, weighted samples, etc.

### Disadvantages
- **Computationally expensive** (though usually fast for modern computers)
- **Requires representative sample**: if the sample is biased, bootstrap results are biased
- **Works poorly with extreme statistics** (e.g., max of a small sample)

In [ ]:
def bootstrap_diff_conversion(df, group1_name, group2_name, n_bootstrap=10000, random_state=42):
    """
    Bootstrap confidence interval for difference in conversion rates: group1 - group2
    
    Parameters:
    -----------
    df : DataFrame
        Data with 'segment' and 'conversion' columns
    group1_name : str
        Name of first group
    group2_name : str
        Name of second group (comparison baseline)
    n_bootstrap : int
        Number of bootstrap samples
    random_state : int
        Random seed
    
    Returns:
    --------
    dict : Contains bootstrap differences, CI, and statistics
    """
    np.random.seed(random_state)
    
    # Get data for each group
    conv1 = df[df['segment'] == group1_name]['conversion'].values
    conv2 = df[df['segment'] == group2_name]['conversion'].values
    
    # Original difference
    original_diff = conv1.mean() - conv2.mean()
    
    # Bootstrap loop
    boot_diffs = []
    for _ in range(n_bootstrap):
        # Resample with replacement
        boot_conv1 = np.random.choice(conv1, size=len(conv1), replace=True)
        boot_conv2 = np.random.choice(conv2, size=len(conv2), replace=True)
        
        boot_diffs.append(boot_conv1.mean() - boot_conv2.mean())
    
    boot_diffs = np.array(boot_diffs)
    
    # Calculate percentile CI (2.5th to 97.5th)
    ci_lower = np.percentile(boot_diffs, 2.5)
    ci_upper = np.percentile(boot_diffs, 97.5)
    
    # Calculate BCa (Bias-Corrected and Accelerated) CI
    # Bias correction
    z0 = stats.norm.ppf(np.mean(boot_diffs < original_diff))
    
    # Acceleration (jackknife)
    jack_diffs = []
    for i in range(len(conv1)):
        conv1_jack = np.delete(conv1, i)
        conv2_sample = conv2.copy()
        jack_diffs.append(conv1_jack.mean() - conv2_sample.mean())
    
    for i in range(len(conv2)):
        conv1_sample = conv1.copy()
        conv2_jack = np.delete(conv2, i)
        jack_diffs.append(conv1_sample.mean() - conv2_jack.mean())
    
    jack_diffs = np.array(jack_diffs)
    jack_mean = np.mean(jack_diffs)
    
    num = np.sum((jack_mean - jack_diffs) ** 3)
    denom = 6 * (np.sum((jack_mean - jack_diffs) ** 2) ** 1.5)
    accel = num / denom if denom != 0 else 0
    
    # BCa percentiles
    p_lower = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(0.025)) / (1 - accel * (z0 + stats.norm.ppf(0.025))))
    p_upper = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(0.975)) / (1 - accel * (z0 + stats.norm.ppf(0.975))))
    
    bca_lower = np.percentile(boot_diffs, p_lower * 100)
    bca_upper = np.percentile(boot_diffs, p_upper * 100)
    
    return {
        'group1': group1_name,
        'group2': group2_name,
        'original_diff': original_diff,
        'boot_differences': boot_diffs,
        'percentile_ci': (ci_lower, ci_upper),
        'bca_ci': (bca_lower, bca_upper),
        'se': np.std(boot_diffs),
        'n_group1': len(conv1),
        'n_group2': len(conv2)
    }

# Run bootstrap for Men's vs Control and Women's vs Control
print("Running bootstrap analysis...")
boot_mens = bootstrap_diff_conversion(df, "Mens E-Mail", "No E-Mail", n_bootstrap=10000)
boot_womens = bootstrap_diff_conversion(df, "Womens E-Mail", "No E-Mail", n_bootstrap=10000)

print("\n=== Bootstrap Results: Conversion Rate Difference ===\n")

for boot_result in [boot_mens, boot_womens]:
    print(f"{boot_result['group1']} vs {boot_result['group2']}:")
    print(f"  Original difference: {boot_result['original_diff']:.4f}")
    print(f"  Bootstrap SE: {boot_result['se']:.4f}")
    print(f"  Percentile 95% CI: [{boot_result['percentile_ci'][0]:.4f}, {boot_result['percentile_ci'][1]:.4f}]")
    print(f"  BCa 95% CI:        [{boot_result['bca_ci'][0]:.4f}, {boot_result['bca_ci'][1]:.4f}]")
    print()

In [ ]:
# Visualize bootstrap distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, boot_result in enumerate([boot_mens, boot_womens]):
    ax = axes[idx]
    
    # Histogram of bootstrap differences
    ax.hist(boot_result['boot_differences'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    
    # Mark original difference
    ax.axvline(boot_result['original_diff'], color='red', linestyle='--', linewidth=2, label='Observed Diff')
    
    # Mark percentile CI
    ci_lower, ci_upper = boot_result['percentile_ci']
    ax.axvline(ci_lower, color='green', linestyle='-', linewidth=2, label='Percentile 95% CI')
    ax.axvline(ci_upper, color='green', linestyle='-', linewidth=2)
    
    # Mark BCa CI
    bca_lower, bca_upper = boot_result['bca_ci']
    ax.axvline(bca_lower, color='orange', linestyle=':', linewidth=2, label='BCa 95% CI')
    ax.axvline(bca_upper, color='orange', linestyle=':', linewidth=2)
    
    ax.axvline(0, color='black', linestyle=':', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Conversion Rate Difference', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f"{boot_result['group1']} vs {boot_result['group2']}", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb04/nb04_bootstrap_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Bootstrap distributions saved.")

## Bootstrap for Continuous Outcomes: Spend Difference

The bootstrap works equally well for continuous outcomes. Instead of resampling conversion (binary), we resample spend values.

In [ ]:
def bootstrap_diff_continuous(df, metric, group1_name, group2_name, n_bootstrap=10000, random_state=42):
    """
    Bootstrap confidence interval for difference in a continuous metric.
    
    Parameters:
    -----------
    df : DataFrame
        Data with 'segment' column
    metric : str
        Column name of continuous metric (e.g., 'spend')
    group1_name : str
        Name of first group
    group2_name : str
        Name of second group
    n_bootstrap : int
        Number of bootstrap samples
    random_state : int
        Random seed
    
    Returns:
    --------
    dict : Bootstrap results
    """
    np.random.seed(random_state)
    
    # Get metric values for each group
    metric1 = df[df['segment'] == group1_name][metric].values
    metric2 = df[df['segment'] == group2_name][metric].values
    
    # Original difference
    original_diff = metric1.mean() - metric2.mean()
    
    # Bootstrap loop
    boot_diffs = []
    for _ in range(n_bootstrap):
        boot_metric1 = np.random.choice(metric1, size=len(metric1), replace=True)
        boot_metric2 = np.random.choice(metric2, size=len(metric2), replace=True)
        boot_diffs.append(boot_metric1.mean() - boot_metric2.mean())
    
    boot_diffs = np.array(boot_diffs)
    
    ci_lower = np.percentile(boot_diffs, 2.5)
    ci_upper = np.percentile(boot_diffs, 97.5)
    
    return {
        'group1': group1_name,
        'group2': group2_name,
        'metric': metric,
        'original_diff': original_diff,
        'boot_differences': boot_diffs,
        'ci': (ci_lower, ci_upper),
        'se': np.std(boot_diffs),
        'n_group1': len(metric1),
        'n_group2': len(metric2)
    }

# Run bootstrap for spend
print("Running bootstrap for spend difference...\n")
boot_spend_mens = bootstrap_diff_continuous(df, 'spend', "Mens E-Mail", "No E-Mail", n_bootstrap=10000)
boot_spend_womens = bootstrap_diff_continuous(df, 'spend', "Womens E-Mail", "No E-Mail", n_bootstrap=10000)

for boot_result in [boot_spend_mens, boot_spend_womens]:
    print(f"{boot_result['group1']} vs {boot_result['group2']} - Spend:")
    print(f"  Original difference: ${boot_result['original_diff']:.2f}")
    print(f"  Bootstrap SE: ${boot_result['se']:.2f}")
    print(f"  95% CI: [${boot_result['ci'][0]:.2f}, ${boot_result['ci'][1]:.2f}]")
    print()

In [ ]:
# Visualize bootstrap distributions for spend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, boot_result in enumerate([boot_spend_mens, boot_spend_womens]):
    ax = axes[idx]
    
    ax.hist(boot_result['boot_differences'], bins=50, alpha=0.7, color='coral', edgecolor='black')
    
    ax.axvline(boot_result['original_diff'], color='red', linestyle='--', linewidth=2, label='Observed Diff')
    
    ci_lower, ci_upper = boot_result['ci']
    ax.axvline(ci_lower, color='green', linestyle='-', linewidth=2, label='95% CI')
    ax.axvline(ci_upper, color='green', linestyle='-', linewidth=2)
    ax.axvline(0, color='black', linestyle=':', linewidth=1, alpha=0.5)
    
    ax.set_xlabel('Spend Difference ($)', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f"{boot_result['group1']} vs {boot_result['group2']}", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb04/nb04_bootstrap_spend_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## Bootstrap Hypothesis Test

### The Idea
Instead of computing a test statistic under a null hypothesis (like frequentist methods), the bootstrap hypothesis test computes:

> **p-value = proportion of bootstrap samples where the statistic is as extreme as observed**

For testing H₀: difference = 0, we ask: "If there were no difference, how often would we see a difference this large just by chance in resamples?"

### Two Approaches
1. **Percentile p-value**: Does zero fall in the 95% CI?
2. **Permutation bootstrap**: Resample from the combined data under H₀ to get the null distribution

Here we'll use the simpler percentile approach.

In [ ]:
def bootstrap_p_value(boot_differences, null_value=0):
    """
    Calculate bootstrap p-value for a two-tailed test.
    
    Parameters:
    -----------
    boot_differences : array
        Bootstrap resamples of the statistic
    null_value : float
        Hypothesized value under H₀ (default 0)
    
    Returns:
    --------
    float : Two-tailed p-value
    """
    centered = boot_differences - np.mean(boot_differences) + null_value
    p_value = np.mean(np.abs(centered - null_value) >= np.abs(np.mean(boot_differences) - null_value))
    return p_value

print("=== Bootstrap Hypothesis Tests (H₀: difference = 0) ===\n")

for boot_result in [boot_mens, boot_womens]:
    p_val = bootstrap_p_value(boot_result['boot_differences'])
    print(f"{boot_result['group1']} vs {boot_result['group2']}:")
    print(f"  Bootstrap p-value: {p_val:.4f}")
    ci_lower, ci_upper = boot_result['percentile_ci']
    print(f"  CI contains zero: {ci_lower <= 0 <= ci_upper}")
    print()

print("\nInterpretation:")
print("- p < 0.05: Reject H₀, likely a real difference")
print("- p >= 0.05: Fail to reject H₀, no strong evidence of difference")

## Comparison: Analytical vs Bootstrap CIs

### Side-by-Side Comparison
Let's compare the analytical Wald CI with bootstrap percentile CI. Both should be similar for conversion rates on large samples.

In [ ]:
# Compile comparison table
comparison_data = []

for boot_result in [boot_mens, boot_womens]:
    group1, group2 = boot_result['group1'], boot_result['group2']
    
    # Get analytical CI
    g1_data = df[df['segment'] == group1]
    g2_data = df[df['segment'] == group2]
    
    conv1_count = g1_data['conversion'].sum()
    conv2_count = g2_data['conversion'].sum()
    n1, n2 = len(g1_data), len(g2_data)
    
    analytical_diff, analytical_lower, analytical_upper = ci_difference_proportions(
        conv1_count, n1, conv2_count, n2
    )
    
    # Bootstrap CI
    boot_lower, boot_upper = boot_result['percentile_ci']
    
    comparison_data.append({
        'Comparison': f"{group1} vs {group2}",
        'Observed Diff': f"{boot_result['original_diff']:.4f}",
        'Analytical CI': f"[{analytical_lower:.4f}, {analytical_upper:.4f}]",
        'Bootstrap CI': f"[{boot_lower:.4f}, {boot_upper:.4f}]",
        'Width (Analytical)': f"{analytical_upper - analytical_lower:.4f}",
        'Width (Bootstrap)': f"{boot_upper - boot_lower:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n=== Analytical vs Bootstrap Confidence Intervals ===\n")
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('../data/outputs/nb04/nb04_ci_comparison.csv', index=False)

## When Bootstrap is Better than Analytical Methods

### Small Sample Sizes
- Analytical methods assume normality, which may not hold with small n
- Bootstrap gives accurate results without distributional assumptions
- **Example**: Testing on only 50 customers per group

### Non-Normal Data
- Spend data is often skewed (many small purchases, few large ones)
- Bootstrap works on the actual distribution of your data
- Analytical methods assume normality and may be inaccurate

### Complex Statistics
- Analytical formulas don't exist for many statistics (e.g., median, trimmed mean, ratio)
- Bootstrap can estimate the sampling distribution of **any** statistic
- **Example**: Bootstrap CI for 90th percentile spend

### Weighted or Stratified Samples
- Bootstrap easily adapts to complex sampling designs
- Analytical methods require mathematical rework

### Robustness
- Bootstrap is more robust to violations of assumptions
- Always gives reasonable results (though maybe not perfectly calibrated)

### Computational Cost
- Modern computers can do 10,000 bootstrap resamples in milliseconds
- The only real drawback is communicating uncertainty to non-technical audiences

### Best Practice
Use both analytical and bootstrap methods. Agreement suggests robust results; disagreement warrants investigation.

In [ ]:
# Create comprehensive results summary
results_summary = {
    'Metric': ['Conversion Rate - Men\'s vs Control', 'Conversion Rate - Women\'s vs Control',
               'Spend - Men\'s vs Control', 'Spend - Women\'s vs Control'],
    'Observed Difference': [
        f"{boot_mens['original_diff']:.4f}",
        f"{boot_womens['original_diff']:.4f}",
        f"${boot_spend_mens['original_diff']:.2f}",
        f"${boot_spend_womens['original_diff']:.2f}"
    ],
    '95% Bootstrap CI': [
        f"[{boot_mens['percentile_ci'][0]:.4f}, {boot_mens['percentile_ci'][1]:.4f}]",
        f"[{boot_womens['percentile_ci'][0]:.4f}, {boot_womens['percentile_ci'][1]:.4f}]",
        f"[${boot_spend_mens['ci'][0]:.2f}, ${boot_spend_mens['ci'][1]:.2f}]",
        f"[${boot_spend_womens['ci'][0]:.2f}, ${boot_spend_womens['ci'][1]:.2f}]"
    ],
    'Includes Zero/No Effect': [
        'Yes' if boot_mens['percentile_ci'][0] <= 0 <= boot_mens['percentile_ci'][1] else 'No',
        'Yes' if boot_womens['percentile_ci'][0] <= 0 <= boot_womens['percentile_ci'][1] else 'No',
        'Yes' if boot_spend_mens['ci'][0] <= 0 <= boot_spend_mens['ci'][1] else 'No',
        'Yes' if boot_spend_womens['ci'][0] <= 0 <= boot_spend_womens['ci'][1] else 'No'
    ]
}

summary_df = pd.DataFrame(results_summary)
print("\n=== BOOTSTRAP ANALYSIS SUMMARY ===\n")
print(summary_df.to_string(index=False))

# Save detailed results
summary_df.to_csv('../data/outputs/nb04/nb04_bootstrap_results.csv', index=False)
print("\nResults saved to ../data/outputs/nb04/nb04_bootstrap_results.csv")